In [16]:
# !pip install PyMuPDF
# !pip install tf-keras

   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 1.7/1.7 MB 22.7 MB/s eta 0:00:00
   ---------------------------------------- 0.0/350.9 MB ? eta -:--:--
    --------------------------------------- 7.3/350.9 MB 41.2 MB/s eta 0:00:09
   - -------------------------------------- 12.6/350.9 MB 31.5 MB/s eta 0:00:11
   -- ------------------------------------- 19.1/350.9 MB 30.2 MB/s eta 0:00:11
   -- ------------------------------------- 24.4/350.9 MB 28.6 MB/s eta 0:00:12
   --- ------------------------------------ 29.4/350.9 MB 29.1 MB/s eta 0:00:12
   --- ------------------------------------ 30.9/350.9 MB 24.9 MB/s eta 0:00:13
   --- ------------------------------------ 33.6/350.9 MB 25.1 MB/s eta 0:00:13
   --- ------------------------------------ 34.3/350.9 MB 20.2 MB/s eta 0:00:16
   ---- ----------------------------------- 38.0/350.9 MB 20.0 MB/s eta 0:00:16
   ----- ---------------------------------- 44.0/350.9 MB 21.4 M

  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.37.1 requires protobuf<6,>=3.20, but you have protobuf 7.34.1 which is incompatible.


In [1]:
file_name = "48 laws of power.pdf"

In [2]:
import fitz  # PyMuPDF

file_name = "48 laws of power.pdf"

def extract_text_from_pdf(file_path, skip_pages=10):
    doc = fitz.open(file_path)
    text = ""
    
    for i, page in enumerate(doc):
        if i < skip_pages:
            continue
        
        text += page.get_text()
    
    return text

raw_text = extract_text_from_pdf(file_name)

print(raw_text[:1000])

LAW 7 pag e 56 
GET OTHERS TO DO THE WORK FOR YOU, BUT ALWAYS TAKE THE CREDIT 
Use the wisdom, knowledge, and legwork of other people to furtker your own cause. Not only will such assis­
tance save you valuable time and energy, it will give you a godlike aura of efficiency and speed. In the end 
your helpers will be forgotten and you will be remembered. Never do yourself what others can do for you. 
LAW S pag e 62 
MAKE OTHER PEOPLE COME TO YOU-USE BAIT IF NECESSARY 
When you force the other person to act, you are the one in control. It is always better to make your opponent 
come to you, abandoning his own plans in the process. Lure him with fabulous gains-then attack. You hold 
the cards. 
LAW<) page 69 
WIN THROUGH YOUR ACTIONS, NEVER THROUGH ARGUMENT 
Any momentary triumph you think you have gained through argument is really a Pyrrhic victory: The resent­
ment and ill will you stir up is stronger and !asts longer than any momentary change of opinion. It is much 
more powerful to ge

In [3]:
import re

def clean_text(text):
    # Remove multiple spaces
    text = re.sub(r'\s+', ' ', text)

    # Fix spaced letters like "P O W E R" → "POWER"
    text = re.sub(r'(\b[A-Z])\s+(?=[A-Z]\b)', r'\1', text)

    # Remove weird line breaks
    text = text.replace('\n', ' ')

    return text.strip()

cleaned_text = clean_text(raw_text)

print(cleaned_text[:1000])

LAW 7 pag e 56 GET OTHERS TO DO THE WORK FOR YOU, BUT ALWAYS TAKE THE CREDIT Use the wisdom, knowledge, and legwork of other people to furtker your own cause. Not only will such assis­ tance save you valuable time and energy, it will give you a godlike aura of efficiency and speed. In the end your helpers will be forgotten and you will be remembered. Never do yourself what others can do for you. LAW S pag e 62 MAKE OTHER PEOPLE COME TO YOU-USE BAIT IF NECESSARY When you force the other person to act, you are the one in control. It is always better to make your opponent come to you, abandoning his own plans in the process. Lure him with fabulous gains-then attack. You hold the cards. LAW<) page 69 WIN THROUGH YOUR ACTIONS, NEVER THROUGH ARGUMENT Any momentary triumph you think you have gained through argument is really a Pyrrhic victory: The resent­ ment and ill will you stir up is stronger and !asts longer than any momentary change of opinion. It is much more powerful to get others to 

In [4]:
def chunk_text(text, chunk_size=500, overlap=100):
  chunks = []
  start = 0
  text_length = len(text)

  while start < text_length:
    end = start + chunk_size
    chunk = text[start:end]
    chunks.append(chunk)
    start += chunk_size - overlap
  return chunks
chunks = chunk_text(cleaned_text)

print("Number of chunks:", len(chunks))
print("\nFirst chunk:\n", chunks[0])

Number of chunks: 3404

First chunk:
 LAW 7 pag e 56 GET OTHERS TO DO THE WORK FOR YOU, BUT ALWAYS TAKE THE CREDIT Use the wisdom, knowledge, and legwork of other people to furtker your own cause. Not only will such assis­ tance save you valuable time and energy, it will give you a godlike aura of efficiency and speed. In the end your helpers will be forgotten and you will be remembered. Never do yourself what others can do for you. LAW S pag e 62 MAKE OTHER PEOPLE COME TO YOU-USE BAIT IF NECESSARY When you force the other person to


In [5]:
for i in range(3):
    print(f"\n--- Chunk {i} ---\n")
    print(chunks[i])


--- Chunk 0 ---

LAW 7 pag e 56 GET OTHERS TO DO THE WORK FOR YOU, BUT ALWAYS TAKE THE CREDIT Use the wisdom, knowledge, and legwork of other people to furtker your own cause. Not only will such assis­ tance save you valuable time and energy, it will give you a godlike aura of efficiency and speed. In the end your helpers will be forgotten and you will be remembered. Never do yourself what others can do for you. LAW S pag e 62 MAKE OTHER PEOPLE COME TO YOU-USE BAIT IF NECESSARY When you force the other person to

--- Chunk 1 ---

AW S pag e 62 MAKE OTHER PEOPLE COME TO YOU-USE BAIT IF NECESSARY When you force the other person to act, you are the one in control. It is always better to make your opponent come to you, abandoning his own plans in the process. Lure him with fabulous gains-then attack. You hold the cards. LAW<) page 69 WIN THROUGH YOUR ACTIONS, NEVER THROUGH ARGUMENT Any momentary triumph you think you have gained through argument is really a Pyrrhic victory: The resent­ me

In [6]:
def filter_useful_text(text):
    # Remove common useless sections
    useless_patterns = [
        r'copyright.*?\.',
        r'all rights reserved.*?\.',
        r'printed and bound.*?\.',
        r'isbn.*?\d+',
        r'first published.*?\.',
        r'page intentionally left blank',
    ]

    for pattern in useless_patterns:
        text = re.sub(pattern, '', text, flags=re.IGNORECASE)

    return text

filtered_text = filter_useful_text(cleaned_text)

print(filtered_text[:1000])

LAW 7 pag e 56 GET OTHERS TO DO THE WORK FOR YOU, BUT ALWAYS TAKE THE CREDIT Use the wisdom, knowledge, and legwork of other people to furtker your own cause. Not only will such assis­ tance save you valuable time and energy, it will give you a godlike aura of efficiency and speed. In the end your helpers will be forgotten and you will be remembered. Never do yourself what others can do for you. LAW S pag e 62 MAKE OTHER PEOPLE COME TO YOU-USE BAIT IF NECESSARY When you force the other person to act, you are the one in control. It is always better to make your opponent come to you, abandoning his own plans in the process. Lure him with fabulous gains-then attack. You hold the cards. LAW<) page 69 WIN THROUGH YOUR ACTIONS, NEVER THROUGH ARGUMENT Any momentary triumph you think you have gained through argument is really a Pyrrhic victory: The resent­ ment and ill will you stir up is stronger and !asts longer than any momentary change of opinion. It is much more powerful to get others to 

In [13]:
pip install sentence-transformers

Note: you may need to restart the kernel to use updated packages.


In [7]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\LENOVO\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\LENOVO\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [8]:
print("Model loaded successfully")

Model loaded successfully


In [9]:
embeddings = model.encode(chunks[:5])
print("Embeddings working:", len(embeddings))

Embeddings working: 5
